<a href="https://colab.research.google.com/github/Meet5738/Langchain_based-AI_Assistant/blob/main/LangChain_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==0.1.16 langchain-community langchain-groq faiss-cpu sentence-transformers

In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Groq API Key: ··········


In [3]:
from google.colab import files

uploaded = files.upload()

Saving 1 to 1
Saving 2 to 2
Saving 3 to 3
Saving 4 to 4
Saving 5 to 5


In [4]:
from langchain.document_loaders import TextLoader

documents = []

for file_name in uploaded.keys():
    loader = TextLoader(file_name, encoding="utf-8")
    documents.extend(loader.load())

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
from langchain.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever()

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [9]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

In [10]:
print(qa.run("Summarize all the files"))

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


Based on the provided context, it appears to be a series of email headers. Here's a summary of the information:

- The emails are from subscriptions@intelligencepress.com to pallen@enron.com.
- The emails are related to NGI Publications, with the subject being "NGI Publications - Thursday, 14 December 2000".
- The emails were sent on December 13, 2000, at 13:28:00 (PST).
- The emails have a Mime-Version of 1.0, Content-Type of text/plain, and Content-Transfer-Encoding of 7bit.
- The emails were stored in a Notes folder on a computer named "Allen-P" with the file name "pallen.nsf".
- The emails were likely stored in a folder named "\Phillip_Allen_Dec2000\Notes Folders\All documents".


In [11]:
print(qa.run("What important information is in these files?"))

Based on the provided email headers, it appears that the files contain information about an email sent to Phillip Allen at Enron on December 13, 2000. The email is from subscriptions@intelligencepress.com and is related to NGI Publications, which is scheduled for Thursday, December 14, 2000.

The important information in these files is likely to be the email content, which is not provided in the headers. However, based on the context, it seems that the email is related to a publication or newsletter from Intelligence Press, and it may contain information about the content, subscription details, or other relevant information for Phillip Allen at Enron.

It's worth noting that the email headers also contain some metadata, such as the sender's and recipient's email addresses, the date and time the email was sent, and the folder and file names where the email was stored. This information can be useful for tracking the email's history and provenance.


In [12]:
print(qa.run("Are there any threats mentioned?"))

There are no threats mentioned in the provided email context.


In [15]:
!pip install -q streamlit pyngrok langchain==0.1.16 langchain-community langchain-groq faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.3 MB/s eta 0:00:00


In [32]:
%%writefile app.py
import streamlit as st
import os

from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_groq import ChatGroq

st.set_page_config(page_title="AI Cyber Assistant", layout="wide")

st.title("AI Cybersecurity Assistant")

groq_key = st.text_input("Enter Groq API Key", type="password")

if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key

# Upload files
uploaded_files = st.file_uploader(
    "Upload Text Files",
    type=["txt"],
    accept_multiple_files=True
)

if uploaded_files and groq_key:
    documents = []

    for file in uploaded_files:
        with open(file.name, "wb") as f:
            f.write(file.read())

        loader = TextLoader(file.name, encoding="utf-8")
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = file.name

        documents.extend(docs)

    st.success(f"{len(uploaded_files)} files uploaded!")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    docs = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings()
    db = FAISS.from_documents(docs, embeddings)
    retriever = db.as_retriever(search_kwargs={"k": 3})

    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0
    )

    query = st.text_input("Ask your question:")

    if query:
        relevant_docs = retriever.get_relevant_documents(query)

        context = "\n\n".join([doc.page_content for doc in relevant_docs])

        prompt = f"""
        Answer based only on the context.

        Context:
        {context}

        Question:
        {query}
        """

        response = llm.invoke(prompt)

        st.subheader("Answer")
        st.write(response.content)

        st.subheader("Sources")

        for i, doc in enumerate(relevant_docs):
            st.markdown(f"**Source {i+1}: {doc.metadata['source']}**")
            st.write(doc.page_content[:300] + "...")
            st.divider()

Overwriting app.py


In [28]:
!ngrok config add-authtoken 2vhabs9whtrqYQ0IAgVyQyA1PA1_5d7JiS11Z7TBXRrfhMrcU

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [33]:
!pkill -f ngrok
!pkill -f streamlit

In [34]:
from pyngrok import ngrok
import subprocess
import time

# Kill previous sessions (important)
!pkill -f streamlit

# Start streamlit in background
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait a bit
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print("App URL:", public_url)

App URL: NgrokTunnel: "https://8fbd-35-238-198-38.ngrok-free.app" -> "http://localhost:8501"
